# Lab 1 — Prepare robot data for OpenVLA

This lab fine-tunes **OpenVLA-7B** (a vision-language-action model: SigLIP + DINOv2 vision, Llama-2 7B
language backbone, actions emitted as 7 discrete tokens) with **LoRA** on a **SageMaker Training Job**.
It follows the workflow of the *Serverless Model Customization* SFT lab (prepare data -> train -> evaluate ->
register -> deploy), but VLAs are not available in serverless customization, so training runs as a
`ModelTrainer` job with our own script, as in the AWS blog
[Fine-tuning OpenVLA on Amazon SageMaker AI with LoRA](https://aws.amazon.com/blogs/physical-ai/fine-tuning-openvla-on-amazon-sagemaker-ai-with-lora/).

In this notebook you:
1. download episodes of [BridgeData V2 (scripted, images)](https://huggingface.co/datasets/VyoJ/BridgeData-V2-Scripted-Images),
2. pseudo-label x/y actions with optical flow (the dataset ships images only),
3. normalize actions to [-1, 1] the way OpenVLA does (per-axis 1st/99th percentile bounds),
4. upload a Hugging Face `DatasetDict` to S3 for the training job.

> **Bring your own data:** replace `bridge_data.pseudo_actions` with real 7-DoF robot actions
> (dx, dy, dz, droll, dpitch, dyaw, gripper). Everything downstream stays the same.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE} -> {config.TRAIN['instance_type']}")
print(f"s3 root:  {s3_root}")

## Build the dataset

Each episode has 3 frames, which gives 2 `(image_t, action_t)` samples. The split is by episode so the
validation set only contains unseen episodes.

The source repo stores 26k individual JPEGs, and loading it file by file hits the Hub's anonymous rate limit
(HTTP 429). `bridge_data.py` reads the Hub's Parquet conversion of the same data instead, from the folder set
in `config.SOURCE_PARQUET_DIR`:

* `"sample_data/bridge-60"` (default): the 60-episode sample shipped with the lab (CC-BY-4.0, see `sample_data/README.md`). Good for a smoke test, too small for real results.
* any folder with shards you downloaded yourself (one ~520 MB file covers 4,200 episodes).
* `None`: download from the Hub. Run `huggingface_hub.login()` first if you get HTTP 429.

In [ ]:
import json
from tqdm.auto import tqdm

from bridge_data import build_dataset

print(f"source: {config.SOURCE_PARQUET_DIR or 'Hugging Face Hub'}")
dataset, action_stats = build_dataset(max_episodes=config.MAX_EPISODES, val_fraction=config.VAL_FRACTION,
                                      progress=tqdm, parquet_dir=config.SOURCE_PARQUET_DIR)
dataset

In [ ]:
import numpy as np

train_actions = np.array(dataset["train"]["action"])
print("normalization bounds (pixels of mean optical flow):")
for axis, lo, hi in zip(["x", "y"], action_stats["q01"], action_stats["q99"]):
    print(f"  {axis}: q01={lo:+.3f}  q99={hi:+.3f}")
print("\nnormalized train actions, mean |a| per axis:", np.abs(train_actions).mean(0).round(3))
print("instruction:", dataset["train"][0]["instruction"])
dataset["train"][0]["image"].resize((224, 168))

## Upload to S3

`save_to_disk` writes Arrow files that the training script reads with `load_from_disk`.
`action_stats.json` travels with the data, then with the model, so the endpoint can return un-normalized actions.

In [ ]:
from pathlib import Path

local_dir = Path("data") / config.DATASET_PREFIX
dataset.save_to_disk(str(local_dir))
(local_dir / "action_stats.json").write_text(json.dumps(action_stats, indent=2))

bucket, prefix = dataset_s3_uri[5:].split("/", 1)
files = [f for f in local_dir.rglob("*") if f.is_file()]
for f in tqdm(files, desc="upload"):
    s3.upload_file(str(f), bucket, prefix + str(f.relative_to(local_dir)))
print(f"dataset: {dataset_s3_uri}  ({len(files)} files)")

Continue with **2-fine-tune-vla.ipynb**.